# Chapter 22 – Lab: from a node-breaker grid file to a search space

Each step of the Lab of [Chapter 22](../index.md) has a cell of guidance and an empty cell for your work. This
notebook needs ToOp: run it with ToOp's interpreter or inside its container, from this folder. The module
`grid_pipeline_toop.py` next to it holds a complete solution; use its functions as building blocks or as a check,
and read `solutions.md` only after your own attempt.

The setup cell copies ToOp's `data/grid_node_breaker/grid.xiidm` into a temporary working folder and imports it
(step 1), so that running the lab never rewrites tracked files.

In [ ]:
import tempfile
from pathlib import Path

import grid_pipeline_toop as gp

work = Path(tempfile.mkdtemp(prefix='ch22_'))
data_folder = gp.import_grid(work / 'node_breaker')
print('processed grid folder:', data_folder)
print(sorted(p.name for p in data_folder.iterdir()))

## Step 1 – Import and preprocess

`gp.import_grid` did the import: `prepare_importer_parameters(grid_file, data_folder)` with
`area_settings.cutoff_voltage = 10`, then `convert_file`. Run the preprocessing too, as `run_pipeline` does:
`run_preprocessing(importer_parameters, data_folder, PreprocessParameters(), is_pandapower_net=False)`, or
`gp.funnel(data_folder)`, which calls `load_grid` directly and returns the final `DynamicInformationStats`.

In [ ]:
# your work here

## Step 2 – Inspect the folder

Load every `masks/*.npy` with NumPy and tabulate the number of `True` entries; read `nminus1_definition.json`,
`importer_auxiliary_data.json`, `static_information_stats.json` and `action_set.json` with pandas. Count actions
per `bus_group_id` and compare each station's count with the upper bound $2^{d-1} - d$. `action_set.json` alone
will not do: it stores the stations without their actions, which live as diffs in `action_set_diffs.hdf5`; load
both with `load_action_set(json, hdf5)` and count the expanded actions.

In this notebook, `gp.mask_counts` tabulates the masks and `gp.actions_per_station` counts the actions per
station; `load_action_set` is in `toop_engine_interfaces.stored_action_set`.

In [ ]:
# your work here

## Step 3 – Watch the funnel

Re-run `load_grid(DirFileSystem(str(data_folder)), parameters=PreprocessParameters(), status_update_fn=record)`
with a callback `record(stage, message, *, stats=None)` that appends the stage name and `stats` to a list, then
plot `n_nodes`, `n_branches`, `n_relevant_subs`, `n_actions` and `n_disc_branches` per stage with matplotlib
(`gp.funnel` returns the table; `gp.stage_that_removes_most` names the stage with the largest drop).

In [ ]:
# your work here

## Step 4 – Change preprocessing

Repeat step 3 with `action_set_filter_bsdf_lodf=False`, `action_set_filter_bridge_lookup=False`,
`action_set_clip=2**3` and `electrical_reassignment_limits=ReassignmentLimits(max_reassignments_per_sub=1)`;
record `n_actions` and the wall-clock time (`gp.variants`). Every `load_grid` run rewrites the artifacts of the
folder, so re-run the default before step 5 if you want the folder to hold the default action set.

In [ ]:
# your work here

## Step 5 – Change import rules

Re-import with `relevant_station_rules=RelevantStationRules(min_busbars=3)` and with
`RelevantStationRules(min_connected_branches=2)` (`gp.import_grid(folder, rules=...)`, `gp.import_rules`), and
explain why lowering the import threshold below four adds nothing after preprocessing.

In [ ]:
# your work here

## Step 6 – A larger grid

Repeat steps 1–3 on `create_complex_grid_battery_hvdc_svc_3w_trafo()` (generated and imported as in
`notebooks/example3_e2e_pipeline.ipynb`) and compare its `stats` with those of the node-breaker grid. Then raise
`cutoff_voltage` to 400 kV and reproduce the first item of the "optimizer found nothing" checklist
(`n_relevant_subs == 0`, `n_actions == 0`, and no error). One might expect ToOp's usual 220 kV cutoff to do it,
but this grid's switchable stations are at 380 kV: with 220 kV the importer still keeps 6 relevant substations
and preprocessing 5 with 100 actions.

In this notebook, `gp.import_grid(folder, network, cutoff_voltage=...)` saves the network as `grid.xiidm` first
and imports it with the given cutoff.

In [ ]:
# your work here